# Experiment 5.4 — SNN-native causal WHAT–WHEN fusion

Analysis-only notebook for the finalized `snn_native_what_when_fusion_v1` artifacts. The primary question is whether aligned frozen WHEN spikes improve final letter classification beyond WHAT-only, elapsed-time, reset-WHEN, and parameter-matched linear fusion.


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

repo_root = Path.cwd()
if not (repo_root / "notebooks").is_dir():
    repo_root = repo_root.parent
root = repo_root / "notebooks" / "artifacts" / "experiment_5_4_snn_native_fusion" / "snn_native_what_when_fusion_v1"

runs = pd.read_csv(root / "runs.csv")
histories = pd.read_csv(root / "histories.csv")
ablations = pd.read_csv(root / "ablation_runs.csv")
activity = pd.read_csv(root / "activity_runs.csv")
paired = pd.read_csv(root / "paired_deltas.csv")
manifest = json.loads((root / "manifest.json").read_text(encoding="utf-8"))
manifest


## Primary architecture comparison

The primary metric is test balanced accuracy. Report mean and standard deviation over the five paired seeds.


In [ ]:
summary = (
    runs.groupby(["condition", "fusion_mode"], as_index=False)
    .agg(
        test_ba_mean=("test_balanced_accuracy", "mean"),
        test_ba_sd=("test_balanced_accuracy", "std"),
        test_macro_f1_mean=("test_macro_f1", "mean"),
        val_ba_mean=("val_balanced_accuracy", "mean"),
        train_ba_mean=("train_balanced_accuracy", "mean"),
        params=("parameter_count", "first"),
    )
    .sort_values("test_ba_mean", ascending=False)
)
summary


In [ ]:
order = manifest["conditions"]
plot_df = summary.set_index("condition").loc[order].reset_index()
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(plot_df))
ax.bar(x, plot_df["test_ba_mean"], yerr=plot_df["test_ba_sd"], capsize=4)
ax.set_xticks(x, plot_df["condition"], rotation=25, ha="right")
ax.set_ylabel("Test balanced accuracy")
ax.set_title("Exp5.4 final letter classification")
ax.grid(axis="y", alpha=0.25)
plt.tight_layout()


## Paired deltas from the primary Fusion-LIF model

Positive `delta_test_balanced_accuracy` means `what_when_fusion_lif` beats the named control on the same seed.


In [ ]:
paired_summary = (
    paired.groupby("comparator", as_index=False)
    .agg(
        delta_ba_mean=("delta_test_balanced_accuracy", "mean"),
        delta_ba_sd=("delta_test_balanced_accuracy", "std"),
        wins=("delta_test_balanced_accuracy", lambda x: int((x > 0).sum())),
        delta_macro_f1_mean=("delta_test_macro_f1", "mean"),
    )
    .sort_values("delta_ba_mean", ascending=False)
)
paired_summary


## WHEN alignment attribution

The trained primary checkpoint is evaluated with aligned WHEN, WHEN zeroed, valid timesteps shuffled, and a circular temporal shift. This distinguishes generic spike statistics from correct WHAT/WHEN temporal alignment.


In [ ]:
ablation_summary = (
    ablations.groupby("ablation", as_index=False)
    .agg(
        test_ba_mean=("test_balanced_accuracy", "mean"),
        test_ba_sd=("test_balanced_accuracy", "std"),
        test_macro_f1_mean=("test_macro_f1", "mean"),
        fusion_fr_mean=("fusion_firing_fraction", "mean"),
    )
    .sort_values("test_ba_mean", ascending=False)
)
ablation_summary


In [ ]:
ordered = ablations[ablations["ablation"] == "ordered"].set_index("seed")
alignment_rows = []
for name in ("when_zero", "when_shuffle", "when_circular_shift"):
    control = (
        ablations[ablations["ablation"] == name]
        .groupby("seed", as_index=True)["test_balanced_accuracy"].mean()
    )
    delta = ordered["test_balanced_accuracy"] - control
    alignment_rows.append({
        "ablation": name,
        "aligned_minus_ablation_ba_mean": delta.mean(),
        "aligned_minus_ablation_ba_sd": delta.std(),
        "aligned_wins": int((delta > 0).sum()),
    })
pd.DataFrame(alignment_rows)


## Generalization and Fusion activity

A large train–test gap would indicate that the Fusion head is fitting user-specific trajectories rather than extracting transferable WHAT×WHEN structure. Fusion firing fraction is only defined for LIF conditions; the linear control has no Fusion spikes.


In [ ]:
generalization = runs.assign(
    train_test_gap=runs["train_balanced_accuracy"] - runs["test_balanced_accuracy"]
).groupby("condition", as_index=False).agg(
    train_ba=("train_balanced_accuracy", "mean"),
    val_ba=("val_balanced_accuracy", "mean"),
    test_ba=("test_balanced_accuracy", "mean"),
    train_test_gap=("train_test_gap", "mean"),
)
generalization


In [ ]:
activity_test = activity[activity["split"] == "test"]
activity_test.groupby("condition", as_index=False).agg(
    fusion_firing_fraction_mean=("fusion_firing_fraction", "mean"),
    fusion_firing_fraction_sd=("fusion_firing_fraction", "std"),
)


## Interpretation checklist

The strongest support for the SNN-native fusion hypothesis requires: (1) main > WHAT-only; (2) main > elapsed-time; (3) main > reset-WHEN; (4) main > parameter-matched linear fusion; and (5) aligned WHEN > zero/shuffle/circular-shift ablations. `when_only_lif` is a leakage/identity diagnostic rather than a desired winner.
